# Call the Deployed GCP Agent

This notebook calls the `deploy_gcp` dice agent after it has been deployed to:

- **Cloud Run** through the ADK API server
- **Vertex AI Agent Engine** through the Vertex AI Python SDK

Run the deployment steps in `src/tutorials/deploy_gcp/README.md` first, then fill the deployment values in `.env`.

## 1. Load `.env`

The notebook reads configuration from the repository `.env` file. Copy `.env.example` to `.env` and set the deployment values before running these cells.

In [ ]:
from pathlib import Path
import os

from dotenv import load_dotenv


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / ".env.example").exists():
            return path
    raise RuntimeError("Could not find repository root")


REPO_ROOT = find_repo_root(Path.cwd())
ENV_PATH = REPO_ROOT / ".env"

if not ENV_PATH.exists():
    raise FileNotFoundError(f"Create {ENV_PATH} from .env.example first")

load_dotenv(ENV_PATH, override=True)
print(f"Loaded {ENV_PATH}")

## 2. Shared Configuration

These values come from `.env`:

- `GOOGLE_CLOUD_PROJECT`
- `GOOGLE_CLOUD_LOCATION`
- `STAGING_BUCKET`
- `CLOUD_RUN_SERVICE_URL`
- `AGENT_ENGINE_ID`

In [ ]:
def env(name: str, default: str | None = None) -> str:
    value = os.getenv(name, default)
    if value is None or value == "":
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value


PROJECT_ID = env("GOOGLE_CLOUD_PROJECT")
LOCATION = env("GOOGLE_CLOUD_LOCATION", "us-central1")
STAGING_BUCKET = env("STAGING_BUCKET")
APP_NAME = env("ADK_APP_NAME", "roll_die")
USER_ID = env("ADK_USER_ID", "notebook-user")
PROMPT = env("ADK_PROMPT", "Roll a 6-sided die 3 times")

print({
    "project": PROJECT_ID,
    "location": LOCATION,
    "app_name": APP_NAME,
    "user_id": USER_ID,
    "prompt": PROMPT,
})

## 3. Call the Cloud Run Deployment

The ADK API server requires a session before calling `/run`. This cell creates a session, sends the prompt, and prints the returned events.

If your Cloud Run service allows unauthenticated requests, set `CLOUD_RUN_AUTH=false` in `.env`. Otherwise, the notebook uses `gcloud auth print-identity-token`.

In [ ]:
import os
import json
import uuid
import requests
import subprocess

def get_cloud_run_url() -> str:
    # Prefer env var, else ask gcloud
    url = os.getenv("CLOUD_RUN_SERVICE_URL", "").strip().rstrip("/")
    if url:
        return url
    name = os.getenv("CLOUD_RUN_SERVICE_NAME", "roll-die-agent")
    region = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1")
    args = [
        "gcloud", "run", "services", "describe", name,
        f"--region={region}",
        "--format=value(status.url)",
    ]
    result = subprocess.run(args, capture_output=True, check=True, text=True).stdout.strip()
    return result.rstrip("/")

def get_cloud_run_headers() -> dict:
    headers = {"Content-Type": "application/json"}
    use_auth = os.getenv("CLOUD_RUN_AUTH", "true").lower() not in {"0", "false", "no"}
    if use_auth:
        token = subprocess.run(
            ["gcloud", "auth", "print-identity-token"],
            capture_output=True, check=True, text=True
        ).stdout.strip()
        headers["Authorization"] = f"Bearer {token}"
    return headers

def extract_response_text(events: list[dict]) -> str:
    # Find last event with text
    for event in reversed(events):
        for part in (event.get("content", {}) or {}).get("parts", []):
            if isinstance(part, dict) and "text" in part:
                if part["text"]:
                    return part["text"]
    return ""

def call_cloud_run(prompt: str) -> list[dict]:
    url = get_cloud_run_url()
    headers = get_cloud_run_headers()
    session_id = f"notebook-{uuid.uuid4().hex[:8]}"

    # Create session
    r = requests.post(
        f"{url}/apps/{APP_NAME}/users/{USER_ID}/sessions/{session_id}",
        headers=headers, json={}, timeout=30
    )
    r.raise_for_status()

    # Run message
    payload = {
        "app_name": APP_NAME,
        "user_id": USER_ID,
        "session_id": session_id,
        "new_message": {"role": "user", "parts": [{"text": prompt}]},
    }
    r2 = requests.post(f"{url}/run", headers=headers, json=payload, timeout=120)
    r2.raise_for_status()
    return r2.json()

cloud_run_events = call_cloud_run(PROMPT)
print(extract_response_text(cloud_run_events))
print(json.dumps(cloud_run_events, indent=2)[:4000])

## 4. Call the Agent Engine Deployment

Set `AGENT_ENGINE_ID` in `.env` to the resource name returned by `adk deploy agent_engine`, then run this cell.

In [ ]:
import vertexai
from vertexai import agent_engines

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

remote_app = agent_engines.get(env("AGENT_ENGINE_ID"))
session = await remote_app.async_create_session(user_id=USER_ID)
session_id = session["id"]
print(f"Agent Engine session: {session_id}")

agent_engine_events = [
    event async for event in remote_app.async_stream_query(
        user_id=USER_ID,
        session_id=session_id,
        message=PROMPT,
    )
]
for event in agent_engine_events:
    print(event)

agent_engine_events

In [ ]:


from pprint import pprint

print("STEP 1: Model request & structure")
print("This event shows the model version, the function calls generated (asking to roll a die 3 times), "
      "and tracks usage, tokens, author, timestamps, and more. "
      "Understanding this dictionary helps us reason about what the agent is actually doing, "
      "which tools it wants to invoke, and how the cost/accounting happens.\n")
pprint(agent_engine_events[0], sort_dicts=False, width=120)
print("\n" + "="*100 + "\n")

print("STEP 2: Function call details (Tool invocation)")
print("The 'parts' field contains the 'function_call' objects. Each models a call to a tool—in this case, "
      "'roll_die' called three times (one per requested roll). "
      "This information makes tool invocations auditable and explainable to humans.\n")
for part in agent_engine_events[0]['content']['parts']:
    if 'function_call' in part:
        pprint(part, sort_dicts=False)
print("\n" + "="*100 + "\n")

print("STEP 3: Function responses (Tool outputs)")
print("After the model requested the function calls, the system responds with actual results. "
      "Each 'function_response' contains the ID tying it back to the correct call, "
      "the tool name, and the output. This structure is essential for both reliability and traceability.\n")
for part in agent_engine_events[1]['content']['parts']:
    if 'function_response' in part:
        pprint(part, sort_dicts=False)
print("\n" + "="*100 + "\n")

print("STEP 4: Usage metadata for accountability")
print("This field tells us how many tokens were used, how many were prompt vs completion, "
      "and modality (text/image/etc). This is critical for cost estimation and troubleshooting model usage.\n")
pprint(agent_engine_events[0].get('usage_metadata', {}), sort_dicts=False)
print("\n" + "="*100 + "\n")

print("STEP 5: Final output as delivered to the user")
print("The last message in the flow shows the final response, in this case the dice results. "
      "This is what the end user receives after all reasoning and tool calls complete.\n")
for part in agent_engine_events[2]['content']['parts']:
    if 'text' in part:
        pprint(part, sort_dicts=False)

## Troubleshooting

- `401` or `403` from Cloud Run usually means your identity token is missing or your account lacks `roles/run.invoker`.
- `404 Session not found` means the session creation request failed or used a different `app_name`, `user_id`, or `session_id` than the `/run` request.
- Agent Engine calls require `AGENT_ENGINE_ID` to be the full resource name, for example `projects/<project-number>/locations/us-central1/reasoningEngines/<engine-id>`.
- If imports fail, run `uv sync --extra adk --extra notebooks` from the repository root and restart the notebook kernel.